# Ekstraksi Data NO2 - Kecamatan Bangkalan (Tugas 3)

Notebook ini menarik data **NO2** dari Sentinel-5P L2 (TROPOMI) melalui
openEO Copernicus Data Space Ecosystem, khusus untuk **Kecamatan
Bangkalan** (bukan lagi seluruh Kabupaten Bangkalan seperti tugas
sebelumnya).

**Perubahan dibanding tugas sebelumnya:**
- AOI diperkecil dari kabupaten (~2.078 km2) menjadi kecamatan (~35 km2).
- Rentang waktu digeser menjadi **31 Agustus 2025 sampai dengan 31 Agustus
  2026** (sebelumnya 24 Agustus).
- Hanya polutan **NO2** yang diambil (sesuai kebutuhan ekstraksi fitur
  TSFEL pada tugas ini).

Keluaran notebook ini adalah file `NO2-Bangkalan.csv`, mengikuti pola
penamaan `<POLUTAN>-<Kecamatan>.csv` seperti pada contoh kode yang
diberikan.


In [1]:
import json
import os
import time

import numpy as np
import openeo
import pandas as pd
import xarray as xr


## 1. Konfigurasi

Polygon di bawah adalah batas administratif **Kecamatan Bangkalan**
(GADM level 3, kode wilayah 3526110), diambil dari sumber data batas
wilayah resmi dan disederhanakan seperlunya. Luasnya sekitar 35 km2,
jauh lebih kecil dari AOI kabupaten yang dipakai pada tugas sebelumnya,
sehingga risiko kena rate limit backend jauh lebih rendah.


In [2]:
BACKEND_URL = "openeo.dataspace.copernicus.eu"

# Koordinat batas Kecamatan Bangkalan (GADM level 3, kode wilayah 3526110)
KECAMATAN_BANGKALAN_COORDS = [
    [112.676376, -7.055057], [112.682755, -7.031777], [112.689285, -7.031271], [112.710609, -7.040352],
    [112.722870, -7.038700], [112.756989, -7.007715], [112.775848, -6.996538], [112.792847, -6.978298],
    [112.806274, -6.989353], [112.801575, -6.997277], [112.790359, -6.999633], [112.785553, -7.011483],
    [112.776878, -7.013729], [112.755203, -7.035440], [112.756142, -7.052607], [112.732155, -7.052158],
    [112.722733, -7.046828], [112.709145, -7.049125], [112.708488, -7.057897], [112.695992, -7.068328],
    [112.676376, -7.055057],
]

kecamatan_polygon = {
    "type": "Polygon",
    "coordinates": [KECAMATAN_BANGKALAN_COORDS],
}

_lons = [pt[0] for pt in KECAMATAN_BANGKALAN_COORDS]
_lats = [pt[1] for pt in KECAMATAN_BANGKALAN_COORDS]
kecamatan_bbox = {
    "west": min(_lons),
    "east": max(_lons),
    "south": min(_lats),
    "north": max(_lats),
}

START_DATE = "2025-08-31"
END_DATE = "2026-08-31"

TARGET_POLLUTANT = "NO2"
KECAMATAN_NAME = "Bangkalan"

NC_DIR = "../data/nc/"
OUTPUT_CSV = f"{TARGET_POLLUTANT}-{KECAMATAN_NAME}.csv"

os.makedirs(NC_DIR, exist_ok=True)

print("Bounding box Kecamatan Bangkalan:", kecamatan_bbox)
print("Jumlah titik polygon:", len(kecamatan_polygon["coordinates"][0]))
print(f"Rentang waktu: {START_DATE} sampai {END_DATE}")
print(f"File keluaran: {OUTPUT_CSV}")


Bounding box Kecamatan Bangkalan: {'west': 112.676376, 'east': 112.806274, 'south': -7.068328, 'north': -6.978298}
Jumlah titik polygon: 21
Rentang waktu: 2025-08-31 sampai 2026-08-31
File keluaran: NO2-Bangkalan.csv


## 2. Autentikasi ke Copernicus Data Space

Sama seperti sebelumnya, memakai device code flow. Buka tautan yang
muncul di output, login, masukkan kode, tunggu sampai terautentikasi.


In [3]:
connection = openeo.connect(BACKEND_URL)
connection = connection.authenticate_oidc()
print("Berhasil terhubung ke:", BACKEND_URL)


Authenticated using refresh token.
Berhasil terhubung ke: openeo.dataspace.copernicus.eu


## 3. Fungsi Ekstraksi (dengan Pengaman yang Sudah Terbukti)

Fungsi di bawah membawa seluruh perbaikan yang sudah terbukti berhasil
pada tugas sebelumnya:
- `safe_download()` menghindari `PermissionError` di Windows dengan
  menghapus file lama sebelum menulis yang baru.
- Kolom nilai dipilih dengan mencocokkan nama band secara eksplisit
  (menghindari bug salah ambil kolom `feature`/`lat`/`lon`).
- Rentang waktu tetap dipecah menjadi beberapa bagian sebagai pengaman,
  meskipun AOI kecamatan jauh lebih kecil sehingga kemungkinan besar
  satu job utuh pun sudah cukup aman.


In [4]:
def safe_download(job, target_path):
    if os.path.exists(target_path):
        try:
            os.remove(target_path)
        except PermissionError:
            print(f"PERINGATAN: {target_path} masih terkunci. Coba restart kernel.")
            raise
    job.get_results().download_file(target=target_path)
    return target_path


def extract_no2_chunk(connection, label, bbox, polygon, start_date, end_date, output_dir):
    cube = connection.load_collection(
        "SENTINEL_5P_L2",
        spatial_extent=bbox,
        temporal_extent=[start_date, end_date],
        bands=[TARGET_POLLUTANT],
    )
    cube = cube.aggregate_temporal_period(period="day", reducer="mean")
    cube = cube.aggregate_spatial(geometries=polygon, reducer="mean")

    result = cube.save_result(format="netCDF")
    job = result.create_job(title=f"{TARGET_POLLUTANT}_{KECAMATAN_NAME}_{label}")
    job.start_and_wait()

    output_path = os.path.join(output_dir, f"{TARGET_POLLUTANT}_{KECAMATAN_NAME}_{label}.nc")
    safe_download(job, output_path)
    print(f"[{label}] Selesai -> {output_path}")
    return output_path


def nc_to_series(nc_path, band_name):
    with xr.open_dataset(nc_path) as ds:
        df = ds.to_dataframe().reset_index()

    time_col = next((c for c in df.columns if c.lower() == "t" or "time" in c.lower()), df.columns[0])

    if band_name in df.columns:
        value_col = band_name
    else:
        exclude = {"feature", "lat", "lon", "latitude", "longitude", "geometry"}
        candidates = [
            c for c in df.columns
            if c != time_col and c.lower() not in exclude and pd.api.types.is_numeric_dtype(df[c])
        ]
        value_col = candidates[0] if candidates else df.columns[-1]

    series = df.set_index(pd.to_datetime(df[time_col]))[value_col]
    series.name = band_name
    series.index.name = "date"
    return series.sort_index()


def make_date_chunks(start_date, end_date, chunk_months=6):
    chunks = []
    current = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)
    while current < end:
        chunk_end = min(current + pd.DateOffset(months=chunk_months), end)
        chunks.append((current.strftime("%Y-%m-%d"), chunk_end.strftime("%Y-%m-%d")))
        current = chunk_end
    return chunks


print("Fungsi siap.")


Fungsi siap.


## 4. Menjalankan Ekstraksi

Karena AOI Kecamatan Bangkalan jauh lebih kecil dari Kabupaten Bangkalan,
rentang waktu dipecah menjadi **2 bagian per 6 bulan** saja (bukan 4
bagian per 3 bulan seperti tugas sebelumnya) sebagai pengaman ringan.
Jika ternyata tetap aman, kamu bisa mencoba memperbesar `chunk_months`
menjadi 12 (satu job penuh) pada percobaan berikutnya.


In [5]:
DATE_CHUNKS = make_date_chunks(START_DATE, END_DATE, chunk_months=6)
print(f"Dibagi menjadi {len(DATE_CHUNKS)} bagian:")
for i, (s, e) in enumerate(DATE_CHUNKS, start=1):
    print(f"  Bagian {i}: {s} sampai {e}")


Dibagi menjadi 3 bagian:
  Bagian 1: 2025-08-31 sampai 2026-02-28
  Bagian 2: 2026-02-28 sampai 2026-08-28
  Bagian 3: 2026-08-28 sampai 2026-08-31


In [6]:
nc_paths = []
for i, (chunk_start, chunk_end) in enumerate(DATE_CHUNKS, start=1):
    label = f"bagian{i}"
    try:
        nc_path = extract_no2_chunk(
            connection=connection,
            label=label,
            bbox=kecamatan_bbox,
            polygon=kecamatan_polygon,
            start_date=chunk_start,
            end_date=chunk_end,
            output_dir=NC_DIR,
        )
        nc_paths.append(nc_path)
    except Exception as exc:
        print(f"[{label}] GAGAL: {exc}")
    time.sleep(15)

print()
print(f"Berhasil {len(nc_paths)} dari {len(DATE_CHUNKS)} bagian.")


0:00:00 Job 'j-260913093536486b90cb76efed91a3e1': send 'start'
0:00:02 Job 'j-260913093536486b90cb76efed91a3e1': queued (progress 0%)
0:00:08 Job 'j-260913093536486b90cb76efed91a3e1': queued (progress 0%)
0:00:15 Job 'j-260913093536486b90cb76efed91a3e1': queued (progress 0%)
0:00:23 Job 'j-260913093536486b90cb76efed91a3e1': queued (progress 0%)
0:00:33 Job 'j-260913093536486b90cb76efed91a3e1': queued (progress 0%)
0:00:45 Job 'j-260913093536486b90cb76efed91a3e1': running (progress N/A)
0:01:01 Job 'j-260913093536486b90cb76efed91a3e1': running (progress N/A)
0:01:20 Job 'j-260913093536486b90cb76efed91a3e1': running (progress N/A)
0:01:44 Job 'j-260913093536486b90cb76efed91a3e1': running (progress N/A)
0:02:15 Job 'j-260913093536486b90cb76efed91a3e1': running (progress N/A)
0:02:52 Job 'j-260913093536486b90cb76efed91a3e1': finished (progress 100%)
[bagian1] Selesai -> ../data/nc/NO2_Bangkalan_bagian1.nc
0:00:00 Job 'j-260913093859493381a0691d4110ffe3': send 'start'
0:00:04 Job 'j-2609130

## 5. Menggabungkan dan Menyimpan

Nilai persis 0.0 tetap diperlakukan sebagai data hilang (NaN), karena
konsentrasi NO2 secara fisis tidak pernah benar benar nol (lihat catatan
pada dokumen Data Understanding).


In [7]:
parts = [nc_to_series(path, TARGET_POLLUTANT) for path in nc_paths]
combined = pd.concat(parts)
combined = combined[~combined.index.duplicated(keep="first")]
combined = combined.sort_index()

df_final = combined.reset_index()
df_final.columns = ["date", TARGET_POLLUTANT]

n_zero = (df_final[TARGET_POLLUTANT] == 0).sum()
df_final[TARGET_POLLUTANT] = df_final[TARGET_POLLUTANT].where(df_final[TARGET_POLLUTANT] != 0)

print(f"Nilai 0.0 dikonversi menjadi NaN: {n_zero}")
print(f"Total baris: {len(df_final)}")
print(f"Jumlah valid: {df_final[TARGET_POLLUTANT].notna().sum()}")
print(f"Jumlah NaN  : {df_final[TARGET_POLLUTANT].isna().sum()}")

df_final.to_csv(OUTPUT_CSV, index=False)
print(f"\nDisimpan -> {OUTPUT_CSV}")
df_final.head(10)


Nilai 0.0 dikonversi menjadi NaN: 0
Total baris: 132
Jumlah valid: 132
Jumlah NaN  : 0

Disimpan -> NO2-Bangkalan.csv


,date,NO2
0,2025-09-03,0.000009
1,2025-09-06,0.000051
2,2025-09-07,0.000013
3,2025-09-12,0.000024
4,2025-09-13,0.000020
5,2025-09-14,0.000022
6,2025-09-15,0.000018
7,2025-09-22,0.000022
8,2025-09-23,0.000042
9,2025-09-24,0.000040


## Selesai

File `NO2-Bangkalan.csv` sudah siap dipakai pada notebook berikutnya
(`2-preprocessing-ekstraksi-fitur.ipynb`) untuk deteksi outlier,
imputasi missing value, dan ekstraksi 68 fitur TSFEL.
